In [1]:
import sys
sys.path.append('../')
import time

import numpy as np
import matplotlib.pyplot as plt

import epics
from siriuspy.devices import CAXCtrl
from caxscripts.h5file import HDF5File

/home/ABTLUS/juventino.fonseca/repos/cax-scripts/examples/../caxscripts/libbeamanalysis.py:6: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 1.23.0)
  from scipy.optimize import curve_fit


In [2]:
cax = CAXCtrl()

# Functions

Useful functions for the scan

In [65]:
def current_position():
    return [cax.slit_A1.top_pos,
            cax.slit_A1.bottom_pos,
            cax.slit_A1.left_pos,
            cax.slit_A1.right_pos]

def open_all_slits():
    cax.slit_A1.move_robust_top(value=19.7-0.04)
    cax.slit_A1.move_robust_bottom(value=35.8)
    cax.slit_A1.move_robust_left(value=43.6-0.04)
    cax.slit_A1.move_robust_right(value=47.2)

def move_robust_all(top_pos,bottom_pos,left_pos,right_pos):
    cax.slit_A1.move_robust_top(value=top_pos)
    cax.slit_A1.move_robust_bottom(value=bottom_pos)
    cax.slit_A1.move_robust_left(value=left_pos)
    cax.slit_A1.move_robust_right(value=right_pos)

# Scan

## Parameters

In [ ]:
top_pos_open = 19.7 - 0.04
bottom_pos_open = 35.8
left_pos_open = 43.6 - 0.04
right_pos_open = 47.2

rangex = 1.3 + 0.1
rangey = 4.8

proport = rangey/rangex

In [39]:
L = 0.4

Ndxs = 7
Ndys = int(proport*Ndxs)
dxs = np.linspace(0,rangex-L,Ndxs)
dys = np.linspace(0,rangey-L,Ndys)

In [66]:
print('N of points:',Ndys*Ndxs)
# print(f'scan time estimative: {Ndys*Ndxs*4/60:.3f} min')

N of points: 161


# Execution

Initial state

In [ ]:
top0, bottom0, left0, right0 = current_position()

with open('initial_position.txt','w') as f:
    f.write(f'top: {top0}\n')
    f.write(f'bottom: {bottom0}\n')
    f.write(f'left: {left0}\n')
    f.write(f'right: {right0}')

print(top0, bottom0, left0, right0)

18.6 35.5 43.5 47.0


Initializate file

In [42]:
filename = 'square_scan_20250716_01.h5'
filedir = "/home/ids/data/2025-07-16-Slits_check"
file = HDF5File(filename=filename,filedir=filedir)

file.save_metadata(metadata_dict={
    'exposure_time': cax.dvf_B1.exposure_time,
    'slit_top': cax.slit_A1.top_pos,
    'slit_bottom': cax.slit_A1.bottom_pos,
    'slit_left': cax.slit_A1.left_pos,
    'slit_right': cax.slit_A1.right_pos
})

Loop

In [ ]:
t0 = time.time()

for i, dy in enumerate(dys):

    move_robust_all(top_pos    = top_pos_open-rangey+L+dy,
                    bottom_pos = bottom_pos_open-dy,
                    left_pos   = left_pos_open,
                    right_pos  = right_pos_open-rangex+L)

    for j, dx in enumerate(dxs):

        print(f'scan step x {j+1}/{len(dxs)} and step y {i+1}/{len(dys)}')

        cax.slit_A1.move_robust_left(value=left_pos_open-dx)
        cax.slit_A1.move_robust_right(value=right_pos_open-rangex+L+dx)

        img1 = cax.dvf_A1.image
        img2 = cax.dvf_B1.image

        pos_dict = {
            'slit_top'    : cax.slit_A1.top_pos,
            'slit_bottom' : cax.slit_A1.bottom_pos,
            'slit_left'   : cax.slit_A1.left_pos,
            'slit_right'  : cax.slit_A1.right_pos
        }

        file.save_group(grpname=f'scan-{i:02d}-{j:02d}',grpmetadata=pos_dict)
        file.save_dataset(dsetname=f'dvf1-img',grpname=f'scan-{i}-{j}',dsetdata=img1)
        file.save_dataset(dsetname=f'dvf2-img',grpname=f'scan-{i}-{j}',dsetdata=img2)

move_robust_all(top_pos    = top0,
                bottom_pos = bottom0,
                left_pos   = left0,
                right_pos  = right0)

t1 = time.time()
print(f'ellapsed time [s]: {t1-t0}')